# 02 — Baseline Sequential RAG Evaluation
Build FAISS index, evaluate 200 questions per dataset, record T_sequential.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [ ]:
import json, time
import pandas as pd
from config import ACCURACY_DIR, INDEX_DIR, LOG_DIR, CORPUS_DIR

os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

## Build Sequential Index (corpus_1000) + Record T_sequential

In [ ]:
from pipeline.chunker import chunk_documents
from pipeline.embedder import embed_chunks, get_model
from pipeline.indexer import build_index, save_index

with open(os.path.join(CORPUS_DIR, 'corpus_1000.json')) as f:
    corpus = json.load(f)

t_start = time.perf_counter()
chunks     = chunk_documents(corpus)
embeddings = embed_chunks(chunks)
index      = build_index(embeddings)
t_seq      = time.perf_counter() - t_start

print(f'T_sequential (1k docs, {len(chunks)} chunks): {t_seq:.2f}s')

# save
index_prefix = os.path.join(INDEX_DIR, 'baseline_1k')
save_index(index, chunks, index_prefix)

# log T_sequential
latency_log = {'corpus_size': 1000, 'n_chunks': len(chunks), 'T_sequential_s': t_seq}
with open(os.path.join(LOG_DIR, 'T_sequential.json'), 'w') as f:
    json.dump(latency_log, f)
print('Index and latency log saved.')

## Gemini Setup

In [ ]:
# Set your API key here or via env var GEMINI_API_KEY
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'YOUR_KEY_HERE')

from pipeline.generator import GeminiGenerator
generator = GeminiGenerator(api_key=GEMINI_API_KEY)

## Evaluate on FinQA (Exact Match)

In [ ]:
from pipeline.baseline import run_baseline
from pipeline.embedder import get_model
from evaluation.metrics import evaluate_dataset

embed_model = get_model()

with open(os.path.join(ACCURACY_DIR, 'finqa_eval_questions.json')) as f:
    finqa_eval = json.load(f)

def baseline_fn(q):
    return run_baseline(q, index, chunks, embed_model, generator)

finqa_results = evaluate_dataset(
    finqa_eval,
    baseline_fn,
    log_path=os.path.join(ACCURACY_DIR, 'baseline_finqa.jsonl')
)
print('FinQA Baseline:', finqa_results)

## Evaluate on MultiHop-RAG (F1)

In [ ]:
with open(os.path.join(ACCURACY_DIR, 'multihop_eval_questions.json')) as f:
    multihop_eval = json.load(f)

multihop_results = evaluate_dataset(
    multihop_eval,
    baseline_fn,
    log_path=os.path.join(ACCURACY_DIR, 'baseline_multihop.jsonl')
)
print('MultiHop Baseline:', multihop_results)

## Save Summary

In [ ]:
summary = {
    'finqa_em':      finqa_results['em'],
    'multihop_f1':   multihop_results['f1'],
    'finqa_latency_ms':    finqa_results['avg_latency_ms'],
    'multihop_latency_ms': multihop_results['avg_latency_ms'],
}
with open(os.path.join(ACCURACY_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved baseline_results.json')
print(summary)